In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import warnings
warnings.filterwarnings("ignore")

pio.renderers.default = 'notebook'

AttributeError: `np.unicode_` was removed in the NumPy 2.0 release. Use `np.str_` instead.

## Load and Prepare Data

In [ ]:
us_states = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas',
    'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware',
    'DC': 'District of Columbia', 'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii',
    'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa',
    'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine',
    'MD': 'Maryland', 'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota',
    'MS': 'Mississippi', 'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska',
    'NV': 'Nevada', 'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico',
    'NY': 'New York', 'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio',
    'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island',
    'SC': 'South Carolina', 'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas',
    'UT': 'Utah', 'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington',
    'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming'
}
states_to_abbr = {v: k for k, v in us_states.items()}

# Load cleaned merged data exported from DataCleaning&Wrangling.ipynb
merged = pd.read_csv('Clean Data/merged_data.csv')
merged['state_abbr'] = merged['state'].map(states_to_abbr)

# Per-capita normalized variables
merged['rd_per_capita'] = merged['research_spending'] / merged['population'] * 1_000_000  # $ per person
merged['f500_per_million'] = merged['f500_count'] / (merged['population'] / 1_000_000)   # F500 per million people

print(f"Loaded: {merged.shape[0]} rows, {merged.state.nunique()} states, {merged.year.min()}-{merged.year.max()}")
merged[['state', 'year', 'gdp_per_capita', 'rd_per_capita', 'f500_per_million']].head()

## Interactive Choropleth Map with Year Slider

In [ ]:
import plotly.graph_objects as go

# Prepare formatted columns
merged['gdp_billions'] = merged['gdp'] / 1e9
merged['gdp_per_capita_k'] = merged['gdp_per_capita'] / 1000
merged['research_spending_m'] = merged['research_spending'] / 1000

years = sorted(merged['year'].unique())

metrics = {
    'GDP per Capita ($)': {
        'col': 'gdp_per_capita',
        'colorscale': 'Viridis',
        'hoverfmt': ':$,.0f'
    },
    'Total GDP ($B)': {
        'col': 'gdp_billions',
        'colorscale': 'Plasma',
        'hoverfmt': ':$,.1f'
    },
    'F500 Companies': {
        'col': 'f500_count',
        'colorscale': 'Inferno',
        'hoverfmt': ':,.0f'
    },
    'R&D Spending ($M)': {
        'col': 'research_spending_m',
        'colorscale': 'Cividis',
        'hoverfmt': ':,.0f'
    },
}

metric_names = list(metrics.keys())
default_metric = metric_names[0]
default_year = years[0]

fig = go.Figure()

# Keep track of trace index for each (metric, year)
trace_map = {}
trace_index = 0

for metric_name, info in metrics.items():
    col = info['col']
    zmin = merged[col].quantile(0.02)
    zmax = merged[col].quantile(0.98)

    for yr in years:
        yr_data = merged[merged['year'] == yr]

        visible = (metric_name == default_metric and yr == default_year)

        fig.add_trace(go.Choropleth(
            locations=yr_data['state_abbr'],
            z=yr_data[col],
            locationmode='USA-states',
            colorscale=info['colorscale'],
            zmin=zmin,
            zmax=zmax,
            text=yr_data['state'],
            visible=visible,
            colorbar_title=metric_name,
            hovertemplate=(
                '<b>%{text}</b><br>' +
                f'{metric_name}: %{{z{info["hoverfmt"]}}}<extra></extra>'
            )
        ))

        trace_map[(metric_name, yr)] = trace_index
        trace_index += 1

num_traces = len(fig.data)

def visible_array(active_metric, active_year):
    arr = [False] * num_traces
    arr[trace_map[(active_metric, active_year)]] = True
    return arr

# Slider: starts with default metric and changes year
slider_steps = []
for yr in years:
    slider_steps.append(dict(
        method='update',
        label=str(yr),
        args=[
            {'visible': visible_array(default_metric, yr)},
            {'title': f'State Economic Indicators — {default_metric} ({yr})'}
        ]
    ))

sliders = [dict(
    active=0,
    currentvalue={'prefix': 'Year: '},
    pad={'t': 30},
    steps=slider_steps
)]

# Dropdown: starts with default year and changes metric
dropdown_buttons = []
for metric_name in metric_names:
    dropdown_buttons.append(dict(
        method='update',
        label=metric_name,
        args=[
            {'visible': visible_array(metric_name, default_year)},
            {'title': f'State Economic Indicators — {metric_name} ({default_year})'}
        ]
    ))

updatemenus = [dict(
    buttons=dropdown_buttons,
    direction='down',
    showactive=True,
    x=0.02,
    xanchor='left',
    y=1.15,
    yanchor='top'
)]

fig.update_layout(
    title=f'State Economic Indicators — {default_metric} ({default_year})',
    geo=dict(
        scope='usa',
        projection_type='albers usa',
        showlakes=True,
        lakecolor='rgb(255,255,255)'
    ),
    sliders=sliders,
    updatemenus=updatemenus,
    height=650,
    margin=dict(l=20, r=20, t=80, b=20),
)

fig.show()

## Animated Bubble Chart (Gapminder-style)
R&D Spending vs GDP per Capita, with bubble size = population and color = F500 count. Animated by year.

In [ ]:
import numpy as np

bubble_df = merged[merged['state'] != 'District of Columbia'].sort_values('year').copy()
bubble_df['year_str'] = bubble_df['year'].astype(str)

fig_bubble = px.scatter(
    bubble_df,
    x='research_spending',
    y='gdp_per_capita',
    size='population',
    color='f500_count',
    hover_name='state',
    animation_frame='year_str',
    animation_group='state',
    size_max=60,
    color_continuous_scale='Inferno',
    labels={
        'research_spending': 'R&D Spending ($K)',
        'gdp_per_capita': 'GDP per Capita ($)',
        'population': 'Population',
        'f500_count': 'F500 Companies',
        'year_str': 'Year',
    },
    title='State Economic Performance: R&D Spending vs GDP per Capita (2006–2023)',
)

fig_bubble.update_layout(
    height=650,
    xaxis=dict(
        type='log',
        title='R&D Spending ($K, log scale)',
        range=[np.log10(100), np.log10(800000)],
    ),
    yaxis=dict(
        range=[25000, bubble_df['gdp_per_capita'].max() * 1.05],
    ),
    coloraxis_colorbar=dict(title='F500<br>Companies'),
    margin=dict(l=60, r=20, t=80, b=60),
)

fig_bubble.update_traces(
    marker=dict(opacity=0.7, line=dict(width=0.5, color='white')),
)

# Slow down the animation
fig_bubble.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 900
fig_bubble.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 400

fig_bubble.show()

## Correlation Heatmap
Pearson correlations between all quantitative variables.

In [ ]:
import plotly.graph_objects as go

corr_cols = ['gdp_per_capita', 'gdp_1000000', 'population', 
             'research_spending', 'rd_per_capita',
             'f500_count', 'f500_per_million']
corr_labels = ['GDP per Capita', 'GDP ($M)', 'Population', 
               'R&D Spending', 'R&D per Capita',
               'F500 Count', 'F500 per Million']

corr_matrix = merged[corr_cols].corr()

annotations = []
for i, row_label in enumerate(corr_labels):
    for j, col_label in enumerate(corr_labels):
        val = corr_matrix.iloc[i, j]
        annotations.append(dict(
            x=col_label, y=row_label,
            text=f'{val:.2f}',
            showarrow=False,
            font=dict(color='white' if abs(val) > 0.5 else 'black', size=12),
        ))

fig_heatmap = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_labels,
    y=corr_labels,
    colorscale='RdBu_r',
    zmin=-1, zmax=1,
    colorbar_title='Correlation',
    hovertemplate='%{y} vs %{x}<br>r = %{z:.3f}<extra></extra>',
))

fig_heatmap.update_layout(
    title='Pearson Correlation Matrix (Raw + Per-Capita Normalized)',
    height=600,
    width=750,
    xaxis=dict(tickangle=-45),
    annotations=annotations,
    margin=dict(l=130, r=20, t=60, b=120),
)

fig_heatmap.show()

## Linked Brushing Dashboard
Brush (click-drag) on the scatter plot to select states. The bar charts update to show F500 count and R&D spending for the selected states.

In [ ]:
import altair as alt

# Use most recent year for the linked brushing snapshot
latest_year = merged['year'].max()
snap = merged[(merged['year'] == latest_year) & (merged['state'] != 'District of Columbia')].copy()
snap['state_abbr'] = snap['state'].map(states_to_abbr)

# Brush selection on the scatter plot
brush = alt.selection_interval()

# Scatter: GDP per capita vs Population, colored by selection
scatter = alt.Chart(snap).mark_circle(size=80).encode(
    x=alt.X('population:Q', title='Population', scale=alt.Scale(type='log')),
    y=alt.Y('gdp_per_capita:Q', title='GDP per Capita ($)'),
    color=alt.condition(brush, 'f500_count:Q', alt.value('lightgray'),
                        scale=alt.Scale(scheme='inferno')),
    tooltip=['state:N', 'gdp_per_capita:Q', 'population:Q', 'f500_count:Q', 'research_spending:Q'],
).properties(
    width=450, height=350,
    title=f'GDP per Capita vs Population ({latest_year}) — Brush to Select'
).add_params(brush)

# Bar chart: F500 count for selected states
bars_f500 = alt.Chart(snap).mark_bar().encode(
    x=alt.X('f500_count:Q', title='F500 Companies'),
    y=alt.Y('state_abbr:N', title=None, sort='-x'),
    color=alt.condition(brush, alt.value('steelblue'), alt.value('lightgray')),
    tooltip=['state:N', 'f500_count:Q'],
).properties(
    width=250, height=350,
    title='Fortune 500 Count'
).transform_filter(brush)

# Bar chart: R&D spending for selected states
bars_rd = alt.Chart(snap).mark_bar().encode(
    x=alt.X('research_spending:Q', title='R&D Spending ($K)'),
    y=alt.Y('state_abbr:N', title=None, sort='-x'),
    color=alt.condition(brush, alt.value('darkorange'), alt.value('lightgray')),
    tooltip=['state:N', 'research_spending:Q'],
).properties(
    width=250, height=350,
    title='R&D Spending'
).transform_filter(brush)

# Combine: scatter on left, two bar charts stacked on right
dashboard = scatter | (bars_f500 & bars_rd)
dashboard = dashboard.properties(
    title=alt.Title('State Economic Indicators — Linked Brushing Dashboard')
).configure_title(fontSize=16)

dashboard

## Animated Bar Chart: Top vs Bottom States by GDP per Capita
Shows the 10 highest and 10 lowest GDP per capita states side by side, animated by year. Red = top performers, blue = bottom performers.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

bar_df = merged[merged['state'] != 'District of Columbia'].copy()
bar_df['state_abbr'] = bar_df['state'].map(states_to_abbr)
years = sorted(bar_df['year'].unique())

n = 10

# Use two subplots: top states on right, bottom states on left
fig_bars = make_subplots(rows=1, cols=2, subplot_titles=('Bottom 10', 'Top 10'),
                         shared_xaxes=False, horizontal_spacing=0.15)

# Fixed y positions so axis never shifts
y_positions = list(range(n))

def get_frame_data(yr):
    yr_data = bar_df[bar_df['year'] == yr].sort_values('gdp_per_capita')
    bottom = yr_data.head(n).reset_index(drop=True)
    top = yr_data.tail(n).reset_index(drop=True)
    return bottom, top

# Build frames
frames = []
for yr in years:
    bottom, top = get_frame_data(yr)
    frames.append(go.Frame(
        data=[
            go.Bar(
                x=bottom['gdp_per_capita'],
                y=y_positions,
                orientation='h',
                marker_color='rgba(100, 149, 237, 0.7)',
                text=[f"{row.state_abbr}  ${row.gdp_per_capita:,.0f}" for _, row in bottom.iterrows()],
                textposition='inside',
                insidetextanchor='end',
                textfont=dict(color='white', size=11),
                hovertemplate='<b>%{customdata}</b><br>$%{x:,.0f}<extra></extra>',
                customdata=bottom['state'],
                name='Bottom 10',
            ),
            go.Bar(
                x=top['gdp_per_capita'],
                y=y_positions,
                orientation='h',
                marker_color='rgba(205, 92, 92, 0.7)',
                text=[f"{row.state_abbr}  ${row.gdp_per_capita:,.0f}" for _, row in top.iterrows()],
                textposition='inside',
                insidetextanchor='end',
                textfont=dict(color='white', size=11),
                hovertemplate='<b>%{customdata}</b><br>$%{x:,.0f}<extra></extra>',
                customdata=top['state'],
                name='Top 10',
            ),
        ],
        name=str(yr)
    ))

# Initial data
bottom_init, top_init = get_frame_data(years[0])

fig_bars.add_trace(go.Bar(
    x=bottom_init['gdp_per_capita'], y=y_positions, orientation='h',
    marker_color='rgba(100, 149, 237, 0.7)',
    text=[f"{row.state_abbr}  ${row.gdp_per_capita:,.0f}" for _, row in bottom_init.iterrows()],
    textposition='inside', insidetextanchor='end', textfont=dict(color='white', size=11),
    hovertemplate='<b>%{customdata}</b><br>$%{x:,.0f}<extra></extra>',
    customdata=bottom_init['state'], name='Bottom 10',
), row=1, col=1)

fig_bars.add_trace(go.Bar(
    x=top_init['gdp_per_capita'], y=y_positions, orientation='h',
    marker_color='rgba(205, 92, 92, 0.7)',
    text=[f"{row.state_abbr}  ${row.gdp_per_capita:,.0f}" for _, row in top_init.iterrows()],
    textposition='inside', insidetextanchor='end', textfont=dict(color='white', size=11),
    hovertemplate='<b>%{customdata}</b><br>$%{x:,.0f}<extra></extra>',
    customdata=top_init['state'], name='Top 10',
), row=1, col=2)

fig_bars.frames = frames

sliders = [dict(
    active=0,
    currentvalue={"prefix": "Year: "},
    pad={"t": 40},
    steps=[dict(
        method="animate",
        args=[[str(yr)], {"frame": {"duration": 600, "redraw": True}, "mode": "immediate"}],
        label=str(yr)
    ) for yr in years]
)]

fig_bars.update_layout(
    title='Top 10 vs Bottom 10 States by GDP per Capita',
    height=550,
    showlegend=False,
    sliders=sliders,
    updatemenus=[dict(
        type="buttons", showactive=False,
        y=-0.05, x=0.5, xanchor="center",
        buttons=[
            dict(label="&#9654; Play", method="animate",
                 args=[None, {"frame": {"duration": 800, "redraw": True}, "fromcurrent": True}]),
            dict(label="&#9724; Pause", method="animate",
                 args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}])
        ]
    )],
    margin=dict(l=20, r=20, t=60, b=60),
)

# Fix both x-axes and hide y-axis tick labels (state names are inside bars)
fig_bars.update_xaxes(range=[0, 100000], title='GDP per Capita ($)', row=1, col=1)
fig_bars.update_xaxes(range=[0, 100000], title='GDP per Capita ($)', row=1, col=2)
fig_bars.update_yaxes(showticklabels=False, fixedrange=True, range=[-0.5, n - 0.5], row=1, col=1)
fig_bars.update_yaxes(showticklabels=False, fixedrange=True, range=[-0.5, n - 0.5], row=1, col=2)

fig_bars.show()

## Export Charts for Dashboard
Saves each chart as a standalone HTML file in `Dashboard/charts/`. Re-run this cell after changing any visualization above to keep the dashboard up to date.

In [ ]:
import os

export_dir = 'Dashboard/charts'
os.makedirs(export_dir, exist_ok=True)

# Plotly charts — fig, fig_bubble, fig_heatmap, fig_bars are defined above
fig.write_html(f'{export_dir}/choropleth.html', include_plotlyjs='cdn')
fig_bubble.write_html(f'{export_dir}/bubble_chart.html', include_plotlyjs='cdn')
fig_heatmap.write_html(f'{export_dir}/correlation_heatmap.html', include_plotlyjs='cdn')
fig_bars.write_html(f'{export_dir}/animated_bars.html', include_plotlyjs='cdn')

# Altair chart — `dashboard` variable from linked brushing cell
dashboard.save(f'{export_dir}/linked_brushing.html')

print(f"Exported {len(os.listdir(export_dir))} charts to {export_dir}/")
for f_name in sorted(os.listdir(export_dir)):
    print(f"  {f_name} ({os.path.getsize(os.path.join(export_dir, f_name)) / 1024:.0f} KB)")